# Component 2: independent OPERATION evaluation (Colab)

Evaluates the **frozen** final model (`models/`) **once** on the OPERATION window, following the protocol fixed in `docs/OPERATION_EVAL.md` (`operation-eval-v1`).

- Nothing is fitted or changed: no V_ref or scaler refit, no training, no threshold change.
- Inference uses only `models/model_contract.json` (NumPy). `label` and `health_score` are removed before inference.
- **Run once.** The evaluation step only runs after you set `CONFIRM_FINAL_TEST = True` in Step 5, and it refuses to overwrite results.
- OPEN/SHORT outputs are ML predictions, **not** confirmed electrical faults. No relay isolation is implemented. Latency is host CPU, **not ESP32**.

All code comes from `src/tinyml/holdout.py` and `scripts/06_evaluate_operation.py`. This notebook only calls it.

Run the cells in order with **Shift + Enter**. Send screenshots after each step as before.

## Step 1: Get the latest repository code and install dependencies

In [ ]:
# STEP 1: Get the latest component2-tinyml code and install dependencies
import os, sys, subprocess

REPO_URL = "https://github.com/Lisura123/J26-IT-348"
BRANCH = "component2-tinyml"
REPO_DIR = "/content/J26-IT-348"
COMPONENT_DIR = f"{REPO_DIR}/component2-tinyml"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)

COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True, check=True).stdout.strip()
print("Commit:", subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--format=%h %s"],
                                capture_output=True, text=True, check=True).stdout.strip())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                f"{COMPONENT_DIR}/requirements.txt"], check=True)
import numpy as np, pandas as pd, sklearn
print("Python", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__,
      "| scikit-learn", sklearn.__version__)

## Step 2: Import the modules and verify the frozen model (no data is read)

In [ ]:
# STEP 2: Import modules and verify the committed model artifacts
import importlib.util
SRC_DIR = f"{COMPONENT_DIR}/src"
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
from tinyml.validate import file_sha256
from tinyml.train import check_dataset_sha256
from tinyml.holdout import PROTOCOL_VERSION, verify_frozen_model

MODEL_DIR = f"{COMPONENT_DIR}/models"
contract = verify_frozen_model(MODEL_DIR)          # pinned SHA-256 values; stops on any difference
print("Protocol:", PROTOCOL_VERSION)
print("Model   :", contract["model_version"], "| status", contract["status"],
      "| trained at", contract["trained_on"]["git_commit"])
print("Weights :", contract["weights_sha256"])
print("Frozen model verified.")

spec = importlib.util.spec_from_file_location("op_eval", f"{COMPONENT_DIR}/scripts/06_evaluate_operation.py")
op_eval = importlib.util.module_from_spec(spec); spec.loader.exec_module(op_eval)

## Step 3: Upload the dataset
Click **Choose Files** and select `pv_fault_dataset.csv`. The file stays only in this Colab session.

In [ ]:
# STEP 3: Upload pv_fault_dataset.csv
from google.colab import files
uploaded = files.upload()
csv_files = [n for n in uploaded if n.endswith(".csv")]
if not csv_files:
    raise SystemExit("No CSV uploaded. Run this cell again.")
DATA_PATH = os.path.abspath(csv_files[0])
print("Dataset file:", DATA_PATH, "|", os.path.getsize(DATA_PATH), "bytes")

## Step 4: Check the dataset fingerprint (stops if it is not the validated file)

In [ ]:
# STEP 4: SHA-256 check
sha = check_dataset_sha256(file_sha256(DATA_PATH))
print("SHA-256 OK:", sha)

## Step 5: Confirm the one-time final test
The OPERATION window is the independent final test and is evaluated **once**. Only continue when the code commit in Step 1 is the approved one.

To confirm, change `False` to `True` in the cell below, then run it.

In [ ]:
# STEP 5: Explicit confirmation (change False to True by hand)
CONFIRM_FINAL_TEST = False
print("Confirmed:", CONFIRM_FINAL_TEST)

## Step 6: Run the OPERATION evaluation (once)
Writes `operation_eval.json` and `operation_eval.md` to `/content/operation_eval/`. Run this cell **once**; it refuses to overwrite existing results.

In [ ]:
# STEP 6: Independent OPERATION evaluation (frozen model, run once)
if not CONFIRM_FINAL_TEST:
    raise SystemExit("Set CONFIRM_FINAL_TEST = True in Step 5 first.")
SPLITS_PATH = f"{COMPONENT_DIR}/results/splits_runwise_v1.json"
OUT_DIR = "/content/operation_eval"
result, paths = op_eval.run(DATA_PATH, SPLITS_PATH, MODEL_DIR, OUT_DIR,
                            confirm_final_test=True, code_commit=COMMIT)
for p in paths:
    print("wrote", p)

## Step 7: Show the report

In [ ]:
# STEP 7: Show the Markdown report
from IPython.display import Markdown, display
display(Markdown(open(paths[1], encoding="utf-8").read()))

## Step 8: Download the results
Downloads `operation_eval.zip`. **Send it back for review.** Nothing is committed until it is reviewed and approved.

In [ ]:
# STEP 8: Zip and download
import shutil
zip_path = shutil.make_archive("/content/operation_eval", "zip", OUT_DIR)
files.download(zip_path)
print("Downloaded", zip_path)